# AI Warden — เทรนโมเดลของเราเอง (LoRA → GGUF)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mntoyg/AI-Warden/blob/main/notebook/train.ipynb)

**ก่อนเริ่ม:** Runtime → Change runtime type → **T4 GPU** แล้ว Runtime → Run all

notebook นี้อ่านข้อมูลจาก `collect_data.ipynb` (ใน Drive: `warden-model-lab/data/`) แล้ว:

1. ตรวจว่าไฟล์ข้อมูลตรงกับ `manifest.json` ที่ตอนเก็บเขียนไว้ (sha256)
2. fine-tune **Qwen2.5-Coder-1.5B-Instruct** (Apache-2.0) ด้วย LoRA — `SMOKE = True` รันแค่ 30 step (~5 นาที) เพื่อลองท่อทั้งเส้นก่อน
3. วัด loss บน val ก่อน/หลัง และเทียบคำตอบโมเดลเดิมกับโมเดลที่เทรน
4. merge → แปลงเป็น **GGUF `q8_0`** (~1.6 GB, พอดี RTX 3050 4 GB ที่วัดไว้) → เขียน `manifest.json` ที่ AI Warden ตรวจก่อนโหลด
5. บันทึกลง Drive: `warden-model-lab/models/<ชื่อ>/`

จากนั้นบนเครื่อง Windows (โฟลเดอร์โมเดลต้องอยู่**นอก** workspace):

```bash
WARDEN_MODEL_GPU=1 WARDEN_MODEL_MANIFEST=/d/models/warden-coder/manifest.json ./scripts/warden-cli.sh run ./my-project aider-local
```

## 1. Setup

In [ ]:
import os, sys, subprocess, pathlib, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/mntoyg/AI-Warden"
if IN_COLAB:
    if not pathlib.Path("/content/AI-Warden").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/AI-Warden"], check=True)
    os.chdir("/content/AI-Warden/notebook")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers", "datasets", "accelerate", "peft", "trl", "sentencepiece"], check=True)
elif pathlib.Path("notebook/train_utils.py").exists():
    os.chdir("notebook")
sys.path.insert(0, os.getcwd())

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set Runtime > T4 GPU")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then Run all again"

## 2. ตั้งค่า

`SMOKE = True` = ลองท่อทั้งเส้นให้เร็ว (โมเดลที่ได้ยังไม่เก่ง) → ผ่านแล้วตั้ง `SMOKE = False` แล้ว Run all อีกรอบเพื่อเทรนจริง

In [ ]:
SMOKE = True                      # True: 30 steps on 200 examples to test the pipeline
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MODEL_NAME = "warden-coder-1.5b"  # also the folder name in Drive
MAX_LEN = 2048                    # tokens per example; longer ones are dropped, not cut
EPOCHS = 1
LR = 2e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
BATCH, GRAD_ACC = 2, 8            # effective batch 16; fits a T4 with gradient checkpointing

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB = pathlib.Path("/content/drive/MyDrive/warden-model-lab")
else:
    LAB = pathlib.Path(".")
DATA_DIR = LAB / "data"
OUT_DIR = LAB / "models" / (MODEL_NAME + ("-smoke" if SMOKE else ""))
WORK = pathlib.Path("/content/work" if IN_COLAB else "outputs/work")
WORK.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("data:", DATA_DIR, "| output:", OUT_DIR)

## 3. ข้อมูล: ตรวจ sha256 แล้วโหลด

In [ ]:
import train_utils
from datasets import load_dataset
from transformers import AutoTokenizer

ds_manifest = train_utils.verify_dataset(DATA_DIR)
print("dataset OK:", ds_manifest["train"]["records"], "train /", ds_manifest["val"]["records"], "val")
print("by source:", ds_manifest.get("records_by_source"))

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
raw = load_dataset("json", data_files={"train": str(DATA_DIR / "train.jsonl"),
                                       "val": str(DATA_DIR / "val.jsonl")})
raw = raw.select_columns(["messages"])

def n_tokens(ex):
    return {"n_tokens": len(tok.apply_chat_template(ex["messages"], tokenize=True))}

sized = raw.map(n_tokens)
for split in ("train", "val"):
    lens = sorted(sized[split]["n_tokens"])
    if lens:
        print(f"{split}: {len(lens)} examples, median {lens[len(lens)//2]} tokens, max {lens[-1]}")
data = sized.filter(lambda ex: ex["n_tokens"] <= MAX_LEN).remove_columns("n_tokens")
print("kept (<= MAX_LEN):", {k: len(v) for k, v in data.items()})
if SMOKE:
    data["train"] = data["train"].shuffle(seed=13).select(range(min(200, len(data["train"]))))
    data["val"] = data["val"].select(range(min(50, len(data["val"]))))
assert len(data["train"]) > 0, "no training examples left - collect more data or raise MAX_LEN"

## 4. เทรน LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
from trl import SFTConfig, SFTTrainer

# float32 weights + fp16 autocast: a T4 has no bf16, and LoRA weights created in
# fp16 make the gradient scaler fail ("Attempting to unscale FP16 gradients").
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32, device_map={"": 0})
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

cfg = SFTConfig(**train_utils.accepted_kwargs(SFTConfig, dict(
    output_dir=str(WORK / "ckpt"), num_train_epochs=EPOCHS, max_steps=30 if SMOKE else -1,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_ratio=0.03, logging_steps=5, eval_strategy="no", save_strategy="no",
    fp16=True, bf16=False, gradient_checkpointing=True, max_seq_length=MAX_LEN,
    packing=False, report_to="none", seed=13)))
trainer = SFTTrainer(**train_utils.accepted_kwargs(SFTTrainer, dict(
    model=model, args=cfg, train_dataset=data["train"], eval_dataset=data["val"],
    processing_class=tok)))

with model.disable_adapter():
    before = trainer.evaluate()["eval_loss"]
t0 = time.time()
trainer.train()
after = trainer.evaluate()["eval_loss"]
print(f"val loss: base {before:.4f} -> tuned {after:.4f}  ({(time.time() - t0) / 60:.1f} min)")
trainer.model.save_pretrained(str(WORK / "adapter"))
tok.save_pretrained(str(WORK / "adapter"))

## 5. เทียบคำตอบ: โมเดลเดิม vs ที่เทรน (3 ตัวอย่างจาก val)

In [ ]:
def answer(messages, max_new_tokens=200):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

model.eval()
for ex in data["val"].select(range(min(3, len(data["val"])))):
    ask = [m for m in ex["messages"] if m["role"] != "assistant"][:2]
    print("=" * 80, "\n[user]", ask[-1]["content"][:300])
    with model.disable_adapter():
        print("\n[base]", answer(ask)[:500])
    print("\n[tuned]", answer(ask)[:500])

## 6. merge → GGUF q8_0 → manifest

In [ ]:
import gc, shutil
from peft import PeftModel

del trainer, model
gc.collect(); torch.cuda.empty_cache()

merged_dir = WORK / "merged"
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map={"": 0})
merged = PeftModel.from_pretrained(base, str(WORK / "adapter")).merge_and_unload()
merged.save_pretrained(str(merged_dir), safe_serialization=True)
tok.save_pretrained(str(merged_dir))
del base, merged
gc.collect()

llama = WORK / "llama.cpp"
if not llama.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(llama)], check=True)
# The converter uses llama.cpp's own gguf-py; Colab already has torch/transformers.
# Its full requirements file pins a CPU torch, so it is only the fallback.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece", "protobuf"], check=True)
gguf = WORK / f"{MODEL_NAME}-q8_0.gguf"
convert = [sys.executable, str(llama / "convert_hf_to_gguf.py"), str(merged_dir),
           "--outtype", "q8_0", "--outfile", str(gguf)]
if subprocess.run(convert).returncode != 0:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(llama / "requirements" / "requirements-convert_hf_to_gguf.txt")], check=True)
    subprocess.run(convert, check=True)
print("GGUF:", gguf, f"{gguf.stat().st_size / 1e9:.2f} GB")

In [ ]:
final = OUT_DIR / gguf.name
shutil.copy2(gguf, final)
llama_commit = subprocess.run(["git", "-C", str(llama), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()
manifest_path = train_utils.write_model_manifest(
    final, MODEL_NAME, BASE_MODEL, ds_manifest,
    extra={"quantization": "q8_0", "smoke": SMOKE, "max_len": MAX_LEN, "epochs": EPOCHS,
           "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
           "val_loss": {"base": round(before, 4), "tuned": round(after, 4)},
           "llama_cpp_commit": llama_commit})
print(manifest_path.read_text())
print("saved to Drive:", OUT_DIR)

## 7. ต่อไป

1. ถ้ารอบนี้เป็น `SMOKE = True` และทุก cell ผ่าน → ตั้ง `SMOKE = False` แล้ว Run all อีกรอบ (เทรนจริง ~30–90 นาทีตามขนาดข้อมูล)
2. ดาวน์โหลดโฟลเดอร์ `warden-model-lab/models/warden-coder-1.5b/` (มี `.gguf` + `manifest.json`) ไปไว้ในเครื่อง **นอก** workspace เช่น `D:\models\warden-coder\`
3. รันใน AI Warden (Git Bash):

```bash
WARDEN_MODEL_GPU=1 WARDEN_MODEL_MANIFEST=/d/models/warden-coder/manifest.json ./scripts/warden-cli.sh run ./my-project aider-local
```

CLI ตรวจ sha256 ของ `.gguf` กับ manifest ก่อนโหลด — ถ้าไฟล์ไม่ตรงจะปฏิเสธ (exit 78)